# 02 — Pré-processamento e Feature Engineering

Etapa de limpeza, criação de variáveis, split treino/teste e construção do
`ColumnTransformer` (pipelines de pré-processamento) — tudo salvo em disco
para os notebooks de modelagem apenas treinarem os modelos.

In [1]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="viridis")
pd.set_option("display.max_columns", 100)

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

import joblib
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv(ROOT / "data" / "raw" / "Telco-Customer-Churn.csv")
df.shape

(7043, 21)

## 1. Limpeza

- Converter `TotalCharges` para numérico;
- Remover os 11 clientes com `tenure = 0` (sem histórico de cobrança);
- Remover `customerID` (identificador, não é característica preditiva).

In [2]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
antes = df.shape[0]
df = df.dropna(subset=["TotalCharges"]).copy()
print(f"Removidas {antes - df.shape[0]} linhas (tenure = 0). Base final: {df.shape[0]}")

df["churn_bin"] = (df["Churn"] == "Yes").astype(int)
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})  # vira categórica

Removidas 11 linhas (tenure = 0). Base final: 7032


## 2. Feature engineering

Novas variáveis baseadas nos insights da EDA:

| Variável | Ideia |
|---|---|
| `faixa_tenure` | faixas de tempo de casa (0-12, 13-24, 25-48, 49-72 meses) |
| `qtd_servicos` | quantos serviços extras o cliente usa (0–8) |
| `gasto_por_mes` | `TotalCharges / tenure` — gasto médio mensal real |
| `contrato_mensal` | flag: contrato mês a mês |
| `tem_fibra` | flag: internet por fibra óptica |
| `suporte_online` | flag: tem TechSupport |
| `tem_parceiro` | flag: tem Partner/Dependents (estabilidade familiar) |

In [3]:
df["faixa_tenure"] = pd.cut(
    df["tenure"], bins=[0, 12, 24, 48, 72],
    labels=["0-12m", "13-24m", "25-48m", "49-72m"], include_lowest=True,
)

servicos = ["PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
            "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
df["qtd_servicos"] = (df[servicos] == "Yes").sum(axis=1)

df["gasto_por_mes"] = np.where(df["tenure"] > 0, df["TotalCharges"] / df["tenure"],
                               df["MonthlyCharges"])
df["contrato_mensal"] = (df["Contract"] == "Month-to-month").astype(int)
df["tem_fibra"] = (df["InternetService"] == "Fiber optic").astype(int)
df["suporte_online"] = (df["TechSupport"] == "Yes").astype(int)
df["tem_parceiro"] = ((df["Partner"] == "Yes") | (df["Dependents"] == "Yes")).astype(int)

display(df.head())
print("Churn médio por faixa de tenure (%):")
display((df.groupby("faixa_tenure", observed=True)["churn_bin"].mean() * 100).round(1))

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,churn_bin,faixa_tenure,qtd_servicos,gasto_por_mes,contrato_mensal,tem_fibra,suporte_online,tem_parceiro
0,7590-VHVEG,Female,No,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0,0-12m,1,29.850000,1,0,0,1
1,5575-GNVDE,Male,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0,25-48m,3,55.573529,0,0,0,0
2,3668-QPYBK,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,0-12m,3,54.075000,1,0,0,0
3,7795-CFOCW,Male,No,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0,25-48m,3,40.905556,0,0,1,0
4,9237-HQITU,Female,No,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,0-12m,1,75.825000,1,1,0,0


Churn médio por faixa de tenure (%):


faixa_tenure
0-12m     47.7
13-24m    28.7
25-48m    20.4
49-72m     9.5
Name: churn_bin, dtype: float64

## 3. Split estratificado treino/teste (80/20)

A estratificação mantém a proporção de churn nos dois conjuntos. O teste fica
**intocado** até a avaliação final.

In [4]:
X = df.drop(columns=["customerID", "Churn", "churn_bin"])
y = df["churn_bin"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y,
)

print(f"Treino: {X_train.shape[0]} clientes | Teste: {X_test.shape[0]} clientes")
print(f"Churn treino: {y_train.mean():.1%} | Churn teste: {y_test.mean():.1%}")

proc = ROOT / "data" / "processed"
proc.mkdir(parents=True, exist_ok=True)

train = pd.concat([X_train, y_train], axis=1)
test = pd.concat([X_test, y_test], axis=1)
train.to_csv(proc / "train.csv", index=False)
test.to_csv(proc / "test.csv", index=False)
joblib.dump({"X_train": X_train, "X_test": X_test,
             "y_train": y_train, "y_test": y_test}, proc / "splits.joblib")
print("Splits salvos em data/processed/")

Treino: 5625 clientes | Teste: 1407 clientes
Churn treino: 26.6% | Churn teste: 26.6%
Splits salvos em data/processed/


## 4. ColumnTransformer (imputação + escalonamento + encoding)

- **Numéricas:** imputar pela mediana + `StandardScaler` (importante para regressão logística).
- **Categóricas:** imputar pela moda + `OneHotEncoder(handle_unknown="ignore")`.

O objeto é ajustado **apenas no treino** e aplicado ao teste — sem *data leakage*.

In [5]:
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

print("Numéricas:", num_cols)
print("Categóricas:", cat_cols)

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), cat_cols),
])

X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)

feats = preprocessor.get_feature_names_out()
print(f"Matriz transformada: {X_train_t.shape[0]} x {X_train_t.shape[1]} features")
print("Exemplos de features:", feats[:5], "...", feats[-3:])

joblib.dump(preprocessor, ROOT / "models" / "preprocessor.joblib")
print("Pré-processador salvo em models/preprocessor.joblib")

Numéricas: ['tenure', 'MonthlyCharges', 'TotalCharges', 'qtd_servicos', 'gasto_por_mes', 'contrato_mensal', 'tem_fibra', 'suporte_online', 'tem_parceiro']
Categóricas: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'faixa_tenure']
Matriz transformada: 5625 x 56 features
Exemplos de features: ['num__tenure' 'num__MonthlyCharges' 'num__TotalCharges'
 'num__qtd_servicos' 'num__gasto_por_mes'] ... ['cat__faixa_tenure_13-24m' 'cat__faixa_tenure_25-48m'
 'cat__faixa_tenure_49-72m']
Pré-processador salvo em models/preprocessor.joblib


### Checklist concluído

- [x] Limpeza (`TotalCharges`, `tenure = 0`, `SeniorCitizen` categórica)
- [x] 7 novas features criadas
- [x] Split estratificado 80/20 salvo (`data/processed/`)
- [x] Pipeline de pré-processamento salvo (`models/preprocessor.joblib`)

**Próximo passo:** `03_modelagem.ipynb`